# Explore here

In [9]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import *
from imblearn.metrics import specificity_score
import requests
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from pickle import dump
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import classification_report
import re
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

In [ ]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/k-nearest-neighbors-project-tutorial/main/tmdb_5000_movies.csv"
response = requests.get(url).content.decode('utf-8')

file_name_m = '../data/raw/tmdb_5000_movies.csv'

with open(file_name_m, 'w') as temp_file:
    temp_file.writelines(response)

In [ ]:
file_name_m = '../data/raw/tmdb_5000_movies.csv'
dfm = pd.read_csv(file_name_m)
#pd.set_option('display.max_columns', None)
dfm.head()

In [ ]:
url = "https://raw.githubusercontent.com/4GeeksAcademy/k-nearest-neighbors-project-tutorial/main/tmdb_5000_credits.csv"
response = requests.get(url).content.decode('utf-8')

file_name_c = '../data/raw/tmdb_5000_credits.csv'

with open(file_name_c, 'w') as temp_file:
    temp_file.writelines(response)

In [ ]:
file_name_c = '../data/raw/tmdb_5000_credits.csv'
dfc = pd.read_csv(file_name_c)
#pd.set_option('display.max_columns', None)
dfc.head()

In [ ]:
# Eliminamos las columnas del dataframe de películas que nos indica el ejercicio que debemos eliminar

dfm.drop(['budget', 'homepage', 'original_language', 'original_title', 'popularity', 'production_companies', 'production_countries', 'release_date', 'spoken_languages', 'status', 'tagline', 'vote_average', 'vote_count', 'revenue', 'runtime'], axis=1, inplace=True)


In [ ]:
dfm[dfm['title'] == 'Avatar']

In [ ]:
dfc[dfc['title'] == 'Avatar']

In [ ]:
dfm[dfm['id'] == 19995]

In [ ]:
dfc[dfc['movie_id'] == 19995]

Confirmo que title y movie_id/coinciden en ambos dataset, así que elimino la columna id del dataframe de películas (ya que movie_id es más descriptivo) y hago la unión por título.


In [ ]:
# Eliminamos las columnas del dataframe de créditos que nos indica el ejercicio que debemos eliminar

dfm.drop(['id'], axis=1, inplace=True)

In [ ]:
df = pd.merge(dfm, dfc, on='title', how='outer')

In [ ]:
df["genres"] = df["genres"].apply(lambda x: [genre["name"].replace(" ", "") for genre in eval(x)])

In [ ]:
df["keywords"] = df["keywords"].apply(lambda x: [keyword["name"].replace(" ", "") for keyword in eval(x)])

In [ ]:
df["cast"] = df["cast"].apply(lambda x: [actor["name"].replace(" ", "") for actor in eval(x)[:3]])

In [ ]:
df["crew"] = df["crew"].apply(lambda x: "".join([member["name"].replace(" ", "") for member in eval(x) if member["job"] == "Director"]))

In [ ]:
df["overview"] = df["overview"].apply(lambda x: re.sub(r"[^a-zA-Z0-9 ]", "", x))

In [ ]:
df['tags'] = df.apply(lambda row: " ".join(
    map(str, [  
        " ".join(row["genres"]) if isinstance(row["genres"], list) else "",  
        " ".join(row["cast"]) if isinstance(row["cast"], list) else "",  
        row["crew"] if isinstance(row["crew"], str) else "",  
        " ".join(row["keywords"]) if isinstance(row["keywords"], list) else "",  
        row["overview"] if isinstance(row["overview"], str) else ""  
    ])
), axis=1)

In [ ]:
vectorizer = TfidfVectorizer()
df_vec = vectorizer.fit_transform(df["tags"])
model = NearestNeighbors(n_neighbors = 4, algorithm = "brute").fit(df_vec)

In [ ]:
def movie_rec(name):
    movie_index = df[df["title"] == name].index[0]
    distances, indices = model.kneighbors(df_vec[movie_index], n_neighbors=4)
    similar_movies = [df["title"][i] for i in indices[0][indices[0] != movie_index]]
    return similar_movies

In [ ]:
recommendations = movie_rec("How to Train Your Dragon")

In [ ]:
recommendations

['How to Train Your Dragon 2', "Dragon Nest: Warriors' Dawn", "Pete's Dragon"]